# 🃏 Poker Arena — Treino do CNN leitor de carta (nb 11)

**O elo que faltava.** A medição foi clara: o localizador acha a carta em qualquer UI e o crop+upscale entrega uma carta grande e limpa — mas o reader ZNCC (calibrado numa fonte só) leu **0/6** na fonte real do 247. Este notebook treina um **CNN minúsculo** que aprende a ler um crop limpo de carta em **centenas de fontes e estilos** → generaliza pra fonte real.

## O que ele faz (célula única)

1. **Gera dados no voo** (domain randomization): cartas em ~centenas de fontes, decks 2/4 cores, naipes desenhados **e** glifos unicode reais (fontTools), rotação, blur, ruído e **oclusão** (carta sobreposta, mantendo o canto do índice — como as hole cards que se cobrem).
2. **Held-out de fontes**: separa fontes de treino e validação — a acurácia de val mede **generalização pra fonte nunca vista** (proxy do sim→real).
3. **Treina** um CNN de 2-3 conv + duas cabeças (rank 13 × naipe 4), **<1 MB**.
4. **Exporta** `card_reader.onnx` (entrada `crop` 1×3×96×64; saídas `rank`, `suit`) e valida no onnxruntime.

## Como usar

1. **Runtime → GPU** (T4 acelera o treino; roda em CPU também) e rode a célula.
2. Baixe o `card_reader.onnx` que ele gera → coloque em **`backend/models/`**.
3. O [`localize_read.py`](../../backend/poker_arena/vision/localize_read.py) **detecta e usa automático** (sem o arquivo, cai no ZNCC — ausência graciosa).
4. **Meça em tela real**: F2 localiza + este CNN lê os crops. É o teste honesto pra banca.

## Por que isto fecha a leitura de cartas em ≤4s

O CNN lê um crop em **~1 ms** (CPU), então cabe folgado no orçamento — e ataca exatamente o que faltava: **ler a fonte real**, não a sintética. Junto com o localizador 1-classe (agnóstico) e a abstenção (que pega qualquer leitura duvidosa), fecha o pipeline **localiza-barato / lê-trivial**, tudo local, sem VLM no caminho crítico.

> Notebook em **célula única** (1 markdown + 1 code). O canônico (64×96) e a ordem das classes (`23456789TJQKA` / `shdc`) **batem** com o `localize_read.py` — não mude um sem o outro.


In [ ]:
# ================================================================
#  Poker Arena — Treino do CNN LEITOR DE CARTA (o elo que faltava)
# ================================================================
# O reader ZNCC lia ~100% no estilo sintetico mas 0/6 na fonte REAL do 247
# (medido). Este CNN minusculo aprende a ler um CROP LIMPO de carta em MUITAS
# fontes/estilos -> generaliza pra fonte real. Entra no localize_read.py como
# 'card_reader.onnx'. Respira: eu explico e mostro cada passo.

import glob, os, random, subprocess, sys, time

print("🃏  Vou treinar os OLHOS que leem a carta. Primeiro, as ferramentas...\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fonttools", "onnx", "onnxruntime"], check=False)

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFilter, ImageFont
from torch.utils.data import DataLoader, Dataset

RANKS, SUITS = "23456789TJQKA", "shdc"          # ORDEM das classes (bate com localize_read.py)
CANON = (64, 96)                                # (W,H) canonico — bate com localize_read._preprocess
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"👀  Dispositivo: {DEV} | torch {torch.__version__}")

# ---------- fontes (diversidade = generalizacao) ----------
_FONTS = [f for p in ("/usr/share/fonts/**/*.ttf", "/usr/share/fonts/**/*.otf", "C:/Windows/Fonts/*.ttf")
          for f in glob.glob(p, recursive=True)] or [None]
_GLYPH = {"s": "♠", "h": "♥", "d": "♦", "c": "♣"}
_RED, _BLK = (200, 30, 40), (20, 20, 20)
_FOURCOL = {"s": (20, 20, 20), "h": (200, 30, 40), "d": (30, 90, 210), "c": (30, 140, 60)}


def _suit_fonts(fonts):
    try:
        from fontTools.ttLib import TTFont
    except Exception:
        return []

    def _has(fp, ch):
        try:
            t = TTFont(fp, fontNumber=0)
            return any(ord(ch) in tb.cmap for tb in t["cmap"].tables)
        except Exception:
            return False
    return [fp for fp in fonts if fp and all(_has(fp, g) for g in _GLYPH.values())]


random.Random(0).shuffle(_FONTS)
cut = max(1, int(len(_FONTS) * 0.85))
FONTS_TR, FONTS_VA = _FONTS[:cut], _FONTS[cut:] or _FONTS[:1]   # HELD-OUT de fontes = proxy sim->real
SUIT_TR, SUIT_VA = _suit_fonts(FONTS_TR), _suit_fonts(FONTS_VA)
print(f"🔤  Fontes: {len(_FONTS)} (treino {len(FONTS_TR)} / val {len(FONTS_VA)}) | com naipe: {len(SUIT_TR)}+{len(SUIT_VA)}\n")


def _pip(d, suit, cx, cy, s, color):
    if suit == "d":
        d.polygon([(cx, cy - s), (cx + s * 0.72, cy), (cx, cy + s), (cx - s * 0.72, cy)], fill=color)
    elif suit == "h":
        r = s * 0.56
        d.ellipse([cx - s * 0.9, cy - s * 0.75, cx - s * 0.9 + 2 * r, cy - s * 0.75 + 2 * r], fill=color)
        d.ellipse([cx + s * 0.9 - 2 * r, cy - s * 0.75, cx + s * 0.9, cy - s * 0.75 + 2 * r], fill=color)
        d.polygon([(cx - s * 0.98, cy - s * 0.2), (cx + s * 0.98, cy - s * 0.2), (cx, cy + s)], fill=color)
    elif suit == "s":
        r = s * 0.56
        d.ellipse([cx - s * 0.9, cy + s * 0.75 - 2 * r, cx - s * 0.9 + 2 * r, cy + s * 0.75], fill=color)
        d.ellipse([cx + s * 0.9 - 2 * r, cy + s * 0.75 - 2 * r, cx + s * 0.9, cy + s * 0.75], fill=color)
        d.polygon([(cx - s * 0.98, cy + s * 0.2), (cx + s * 0.98, cy + s * 0.2), (cx, cy - s)], fill=color)
        d.polygon([(cx - s * 0.32, cy + s), (cx + s * 0.32, cy + s), (cx + s * 0.1, cy + s * 0.3), (cx - s * 0.1, cy + s * 0.3)], fill=color)
    else:
        r = s * 0.42
        for ox, oy in [(0, -s * 0.45), (-s * 0.5, s * 0.25), (s * 0.5, s * 0.25)]:
            d.ellipse([cx + ox - r, cy + oy - r, cx + ox + r, cy + oy + r], fill=color)
        d.polygon([(cx - s * 0.32, cy + s), (cx + s * 0.32, cy + s), (cx + s * 0.1, cy + s * 0.25), (cx - s * 0.1, cy + s * 0.25)], fill=color)


def _draw_suit(d, suit, cx, cy, s, color, rng, suit_fonts):
    if suit_fonts and rng.random() < 0.55:
        try:
            f = ImageFont.truetype(rng.choice(suit_fonts), max(6, int(s * 2.2)))
            g = _GLYPH[suit]
            bb = d.textbbox((0, 0), g, font=f)
            d.text((cx - (bb[2] - bb[0]) / 2 - bb[0], cy - (bb[3] - bb[1]) / 2 - bb[1]), g, font=f, fill=color)
            return
        except Exception:
            pass
    _pip(d, suit, cx, cy, s, color)


def make_crop(rank, suit, rng, fonts, suit_fonts, size=96):
    W, H = int(size * rng.uniform(0.66, 0.72)), size
    sc = 3
    Ws, Hs = W * sc, H * sc
    im = Image.new("RGB", (Ws, Hs), tuple(rng.randint(20, 70) for _ in range(3)))
    d = ImageDraw.Draw(im)
    m = int(Ws * rng.uniform(0.04, 0.12))
    tint = rng.randint(236, 255)
    d.rounded_rectangle([m, m, Ws - m, Hs - m], radius=int(Ws * 0.1), fill=(tint, tint, rng.randint(233, tint)))
    color = _FOURCOL[suit] if rng.random() < 0.35 else (_RED if suit in "hd" else _BLK)
    fp = rng.choice(fonts)
    fs = int(Hs * rng.uniform(0.17, 0.24))
    try:
        font = ImageFont.truetype(fp, fs) if fp else ImageFont.load_default()
    except Exception:
        font = ImageFont.load_default()
    txt = "10" if (rank == "T" and rng.random() < 0.5) else rank
    tx, ty = m + int(Ws * rng.uniform(0.02, 0.06)), m + int(Hs * rng.uniform(0.02, 0.05))
    d.text((tx, ty), txt, font=font, fill=color)
    _draw_suit(d, suit, tx + int(fs * 0.35), ty + fs + int(fs * 0.45), fs * 0.28, color, rng, suit_fonts)
    if rng.random() < 0.8:
        _draw_suit(d, suit, Ws // 2, int(Hs * 0.58), Ws * rng.uniform(0.16, 0.24), color, rng, suit_fonts)
    if rng.random() < 0.6:
        im = im.rotate(rng.uniform(-6, 6), fillcolor=(40, 40, 40))
    if rng.random() < 0.35:  # oclusao (carta sobreposta), mantendo o canto do indice
        dd = ImageDraw.Draw(im)
        if rng.random() < 0.5:
            dd.rectangle([int(Ws * rng.uniform(0.55, 0.75)), 0, Ws, Hs], fill=tuple(rng.randint(230, 255) for _ in range(3)))
        else:
            dd.rectangle([0, int(Hs * rng.uniform(0.6, 0.82)), Ws, Hs], fill=tuple(rng.randint(230, 255) for _ in range(3)))
    im = im.resize((W, H), Image.LANCZOS)
    if rng.random() < 0.4:
        im = im.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.0)))
    arr = np.clip(np.asarray(im).astype("int16") + np.random.default_rng(rng.randint(0, 1 << 30)).normal(0, rng.uniform(2, 10), (H, W, 3)), 0, 255).astype("uint8")
    return arr


def preprocess(arr_hwc):
    im = Image.fromarray(arr_hwc).convert("RGB").resize(CANON, Image.BILINEAR)
    a = (np.asarray(im, dtype=np.float32) / 255.0 - 0.5) / 0.5
    return a.transpose(2, 0, 1)


class CardDS(Dataset):
    def __init__(self, fonts, suit_fonts, n, base):
        self.fonts, self.suit_fonts, self.n, self.base = fonts, suit_fonts, n, base

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = random.Random((self.base * 2654435761 + i) & 0xFFFFFFFF)
        r, s = rng.randrange(13), rng.randrange(4)
        x = preprocess(make_crop(RANKS[r], SUITS[s], rng, self.fonts, self.suit_fonts))
        return torch.from_numpy(x), r, s


class Reader(nn.Module):
    def __init__(self):
        super().__init__()

        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(3, 16), blk(16, 32), blk(32, 64), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.rank, self.suit = nn.Linear(64, 13), nn.Linear(64, 4)

    def forward(self, x):
        z = self.f(x)
        return self.rank(z), self.suit(z)


print("🏭  Gerando dados no voo e treinando (held-out de fontes = teste de generalizacao)...")
tr = DataLoader(CardDS(FONTS_TR, SUIT_TR, 40000, 1), batch_size=128, shuffle=True, num_workers=2)
va = DataLoader(CardDS(FONTS_VA, SUIT_VA, 4000, 999), batch_size=128, num_workers=2)
model = Reader().to(DEV)
opt = torch.optim.Adam(model.parameters(), 1e-3)
lossf = nn.CrossEntropyLoss()
EPOCHS = 8
for ep in range(EPOCHS):
    model.train()
    t0 = time.time()
    for x, r, s in tr:
        x, r, s = x.to(DEV), r.to(DEV), s.to(DEV)
        pr, ps = model(x)
        loss = lossf(pr, r) + lossf(ps, s)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    rk = sk = ck = tot = 0
    with torch.no_grad():
        for x, r, s in va:
            x = x.to(DEV)
            pr, ps = model(x)
            pr, ps = pr.argmax(1).cpu(), ps.argmax(1).cpu()
            rk += (pr == r).sum().item()
            sk += (ps == s).sum().item()
            ck += ((pr == r) & (ps == s)).sum().item()
            tot += len(r)
    print(f"  época {ep+1}/{EPOCHS} [{time.time()-t0:4.0f}s]  val rank {100*rk/tot:5.1f}%  naipe {100*sk/tot:5.1f}%  CARTA {100*ck/tot:5.1f}%")

# ---------- export ONNX ----------
print("\n📦  Exportando card_reader.onnx...")
model.eval().cpu()
torch.onnx.export(model, torch.zeros(1, 3, 96, 64), "card_reader.onnx",
                  input_names=["crop"], output_names=["rank", "suit"], opset_version=13)
print(f"✅  card_reader.onnx salvo ({os.path.getsize('card_reader.onnx')/1024:.0f} KB).")

# valida o ONNX (bate com o torch?)
import onnxruntime as ort
sess = ort.InferenceSession("card_reader.onnx", providers=["CPUExecutionProvider"])
xb, rb, sb = next(iter(va))
o = sess.run(None, {"crop": xb[:64].numpy()})
acc = ((o[0].argmax(1) == rb[:64].numpy()) & (o[1].argmax(1) == sb[:64].numpy())).mean()
print(f"🔎  ONNX check: acurácia de CARTA num batch val = {100*acc:.1f}%")

try:
    from google.colab import files
    files.download("card_reader.onnx")
except Exception:
    pass
print("\n👉  Baixe card_reader.onnx -> coloque em backend/models/ -> o localize_read.py usa automático.")
print("    Meça em tela real com o harness (F2 localiza + este CNN lê).")
